# Convert 3D TIFF → OME-Zarr (NGFF, multiscale)
**+ serve locally and generate a Neuroglancer URL**  
**Last updated:** 2025-11-09

This beginner-friendly notebook shows how to:
1) Load a **3D TIFF** (Z,Y,X)  
2) Write an **OME-Zarr NGFF** multiscale pyramid  
3) Run a **CORS-enabled local server** to share it  
4) (Option A) Build a **Neuroglancer precomputed** volume from the TIFF and generate a **Neuroglancer URL**  
5) (Option B) Link directly if your Neuroglancer build supports **Zarr/NGFF** (experimental)

> Neuroglancer natively supports the **precomputed** format. For a guaranteed experience, we create a precomputed pyramid for viewing. OME‑Zarr is excellent for analysis & napari; you can keep **both** side-by-side.

## 0) Install tools (one time)
Use **conda/mamba** (recommended) or **pip**:

```bash
# OME-Zarr + TIFF IO + Dask
mamba create -n ngff -c conda-forge python tifffile ome-zarr zarr dask numcodecs scikit-image
mamba activate ngff

# For Neuroglancer precomputed conversion
mamba install -c conda-forge cloud-volume

# (Optional) If you prefer pip
pip install tifffile ome-zarr zarr dask[numexpr] numcodecs scikit-image cloud-volume
```

In [1]:
import os, json
from pathlib import Path
import numpy as np

try:
    import tifffile as tiff
except Exception as e:
    tiff = None
    print('tifffile not available:', e)

try:
    import dask.array as da
except Exception as e:
    da = None
    print('dask not available:', e)

try:
    from ome_zarr.writer import write_image
    from ome_zarr.scale import Scaler
except Exception as e:
    write_image = None
    Scaler = None
    print('ome-zarr not available:', e)

print('Libs:', {'tifffile': tiff is not None, 'dask': da is not None, 'ome-zarr': write_image is not None})


ome-zarr not available: No module named 'ome_zarr'
Libs: {'tifffile': True, 'dask': True, 'ome-zarr': False}


## 1) Load a 3D TIFF (Z,Y,X)
Edit the `tiff_path` to point at your file. We memory-map or Dask-chunk it for large data.

In [ ]:
tiff_path = '/path/to/volume.tif'  # <-- EDIT

if tiff is None:
    raise RuntimeError('Install tifffile to proceed.')

# Load lazily with tifffile (will read into memory when sliced) or convert to Dask
arr_np = tiff.imread(tiff_path)  # expects (Z,Y,X) or (Z,Y,X,C)
if arr_np.ndim == 4 and arr_np.shape[-1] in (1,2,3,4):
    # Move channels first if present: (C,Z,Y,X)
    arr_np = np.moveaxis(arr_np, -1, 0)
print('TIFF shape:', arr_np.shape, 'dtype:', arr_np.dtype)

if da is not None:
    # chunk Z in modest slabs, and tiles in XY (tune per dataset)
    chunks = (1,) + tuple(min(512, s) for s in arr_np.shape[-2:]) if arr_np.ndim==3 else (1,1) + tuple(min(512, s) for s in arr_np.shape[-2:])
    arr = da.from_array(arr_np, chunks=chunks)
else:
    arr = arr_np


## 2) Write OME‑Zarr (NGFF) multiscale
We create a **multiscale pyramid** with 2× downsampling in Z,Y,X per level. The output is a Zarr directory you can serve directly.


In [ ]:
out_ngff = '/path/to/out_ngff.ome.zarr'  # <-- EDIT
Path(out_ngff).mkdir(parents=True, exist_ok=True)

if write_image is None or Scaler is None:
    raise RuntimeError('Install ome-zarr to write NGFF.')

# Define voxel size (optional but recommended); example in micrometers
voxel_size_um = (1.0, 0.5, 0.5)  # (Z, Y, X) spacing; EDIT to match your data

# Set up a pyramidal scaler; 4 levels is a reasonable default for large volumes
scaler = Scaler(scale_factors=[
    [1, 1, 1],    # level 0 (full res)
    [2, 2, 2],    # level 1
    [4, 4, 4],    # level 2
    [8, 8, 8],    # level 3
])

# Construct NGFF metadata
metadata = {
    'name': 'volume',
    'axes': [
        {'name': 'z', 'type': 'space', 'unit': 'micrometer'},
        {'name': 'y', 'type': 'space', 'unit': 'micrometer'},
        {'name': 'x', 'type': 'space', 'unit': 'micrometer'}
    ],
    'coordinateTransformations': [
        [{'type': 'scale', 'scale': [float(voxel_size_um[0]), float(voxel_size_um[1]), float(voxel_size_um[2])]}]
    ]
}

# Write the image; will compute/downsample using Dask if available
write_image(arr, out_ngff, scaler=scaler, axes=('z','y','x'), storage_options={'chunks': True}, metadata=metadata)
print('Wrote NGFF:', out_ngff)


## 3) Serve locally (CORS-enabled)
To let web apps fetch your data from `localhost`, start a tiny CORS server in the **parent directory** of your `.ome.zarr`:

```bash
python cors_server.py --directory /path/to --port 8000
```
Then your NGFF will be reachable at:
`http://127.0.0.1:8000/out_ngff.ome.zarr/`

In [ ]:
# Write the tiny CORS server (reuse if you already have it)
cors_server = r'''from http.server import SimpleHTTPRequestHandler, HTTPServer
class CORSRequestHandler(SimpleHTTPRequestHandler):
    def end_headers(self):
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Range, Content-Type')
        return super().end_headers()

if __name__ == '__main__':
    import argparse, os
    ap = argparse.ArgumentParser()
    ap.add_argument('--port', type=int, default=8000)
    ap.add_argument('--directory', type=str, default='.')
    args = ap.parse_args()
    os.chdir(args.directory)
    httpd = HTTPServer(('0.0.0.0', args.port), CORSRequestHandler)
    print(f'Serving {os.getcwd()} at http://127.0.0.1:{args.port} (CORS enabled)')
    httpd.serve_forever()
'''
with open('/mnt/data/cors_server.py','w',encoding='utf-8') as f:
    f.write(cors_server)
print('Saved: /mnt/data/cors_server.py')


## 4) Neuroglancer **precomputed** (recommended for viewing)
Create a **precomputed** pyramid from the same TIFF using **CloudVolume**. This guarantees compatibility with Neuroglancer. Choose a **dtype** and **encoding** (often `raw` or `jpeg` for 8‑bit).

In [ ]:
from pathlib import Path

tiff_path = tiff_path  # reuse
precomputed_dir = '/path/to/precomputed'  # <-- EDIT
Path(precomputed_dir).mkdir(parents=True, exist_ok=True)

try:
    from cloudvolume import CloudVolume
    import numpy as np
except Exception as e:
    CloudVolume = None
    print('cloud-volume not available:', e)

if CloudVolume is not None:
    # Re-read TIFF to ensure we own dtype
    vol = tiff.imread(tiff_path)
    # Neuroglancer expects (Z,Y,X[,C]); we handle grayscale here
    if vol.ndim == 4 and vol.shape[-1] in (1,2,3,4):
        # Already has channels
        pass
    else:
        # make it 4D with channel dim = 1
        vol = vol[..., np.newaxis]

    # Define voxel size in **nanometers** and offset (edit to match your data)
    voxel_size_nm = [1000, 500, 500]  # (Z,Y,X) in nanometers; EDIT
    info = CloudVolume.create_new_info(
        num_channels=vol.shape[-1],
        layer_type='image',
        data_type=str(vol.dtype),
        encoding='raw',
        resolution=[voxel_size_nm[2], voxel_size_nm[1], voxel_size_nm[0]],  # X,Y,Z order!
        voxel_offset=[0,0,0],
        chunk_size=[64,64,64],  # tune for dataset
        volume_size=[vol.shape[2], vol.shape[1], vol.shape[0]]  # X,Y,Z
    )

    vol_path = 'file://' + precomputed_dir
    cv = CloudVolume(vol_path, info=info, parallel=True, progress=True)
    cv.commit_info()

    # Write level 0
    cv[:,:,:] = vol  # (Z,Y,X,C) written as (X,Y,Z,C)
    print('Wrote precomputed volume at', precomputed_dir)
else:
    print('Install cloud-volume to generate precomputed data.')


## 5) Serve precomputed & build a Neuroglancer URL
Serve the **parent** of your `precomputed` folder with the same CORS server:

```bash
python /mnt/data/cors_server.py --directory /path/to --port 8000
```
Then the dataset will be reachable at:  
`http://127.0.0.1:8000/precomputed/`

In [ ]:
from urllib.parse import quote

ng_layer_url = 'precomputed://http://127.0.0.1:8000/precomputed/'  # <-- EDIT if needed

# Minimal Neuroglancer state (single grayscale image layer)
state = {
  'layers': [
    {
      'type': 'image',
      'name': 'vol',
      'source': ng_layer_url
    }
  ],
  'navigation': {
      'pose': {
          'position': {'voxelSize': [1,1,1], 'voxelCoordinates': [0,0,0]}
      }
  }
}

state_json = json.dumps(state, separators=(',',':'))
encoded = quote(state_json, safe='')

# Neuroglancer URL template
ng_base = 'https://nglancer.org/#!'  # replace with your Neuroglancer deployment if different
full_url = ng_base + encoded
print('Neuroglancer URL:')
print(full_url)


## 6) (Optional) Direct Zarr/NGFF in Neuroglancer (experimental)
Some builds of Neuroglancer can open **zarr://** sources. If yours supports it, try building a state where the layer `source` is:

`zarr://http://127.0.0.1:8000/out_ngff.ome.zarr`

Otherwise, prefer the **precomputed** path above for reliability.

## 7) Checklist & tips
- **Units** matter: NGFF uses your spacing metadata; Neuroglancer needs **nanometer** units in precomputed `info`.
- For **very large** volumes, raise chunk sizes and write multi-resolution levels in precomputed (CloudVolume supports mip levels).
- If the browser blocks loading, check **CORS** headers and file paths.
- Keep NGFF for analysis (napari/NGFF tools) and use **precomputed** specifically for Neuroglancer viewing.